In [18]:
import pandas as pd
import sys
print(sys.executable)
pd.set_option('display.float_format', lambda x: '%.6f' % x)

/workspaces/crypto/.venv/bin/python


In [19]:
df = pd.read_csv('data.csv')

In [20]:
df["month"] = pd.to_datetime(df["Time"]).dt.to_period("M")
df["dia"] = pd.to_datetime(df["Time"]).dt.to_period("D")
df['Amount'] = df['Amount'].str.replace(r'[A-Za-z]+', '', regex=True).astype(float)
df['Fee'] = df['Fee'].str.replace(r'[A-Za-z]+', '', regex=True).astype(float)
df['Executed'] = df['Executed'].str.replace(r'[A-Za-z]+', '', regex=True).astype(float)
df['profit'] = df.apply(lambda row: row['Amount'] - row['Fee'] if row['Side'] == 'SELL' else row['Amount'], axis=1)
df=(df[df['month']>='2024-01'] ).sort_values(['Pair', 'Time']).reset_index(drop=True)



In [21]:
def filtrar_trades(df, data_inicio, data_fim#, side='SELL'
                   ):
    return (
        df[
            df['dia'].between(data_inicio, data_fim) #&
           # (df['Side'] == side)
        ]
        .assign(
            igual_linha_acima=lambda d: (
                #(d['Time'] == d['Time'].shift()) &
                (d['Pair'] == d['Pair'].shift()) &
                (d['Side'] == d['Side'].shift()) &
                (d['Price'] == d['Price'].shift())
            )
        )
        .pipe(
            lambda d: d.groupby((~d['igual_linha_acima']).cumsum()).agg(
                Time=('Time', 'first'),
                Pair=('Pair', 'first'),
                Side=('Side', 'first'),
                dia=('dia', 'first'),
                Price=('Price', 'first'),
                Executed=('Executed', 'sum'),
                Amount=('Amount', 'sum'),
                profit=('profit', 'sum'),
            ).reset_index(drop=True)
        )
    )



In [22]:
df['Pair'].unique()

<StringArray>
[  '1000SATSUSDT', '1MBABYDOGEUSDT',       'AAVEUSDC',        'AMPUSDT',
        'ARBUSDT',        'BCHUSDC',        'BIOUSDC',        'BIOUSDT',
        'BLZUSDT',         'BNBEUR',        'BNBUSDC',        'BNBUSDT',
       'BOMEUSDC',       'CITYUSDT',        'CLVUSDT',       'COTIUSDT',
       'CTSIUSDT',       'CTXCUSDT',       'DOGEUSDT',        'DOTUSDC',
        'DOTUSDT',        'EDUUSDT',        'ENAUSDC',        'ENAUSDT',
         'ETHEUR',      'ETHFIUSDT',        'ETHUSDC',        'EURUSDC',
        'EURUSDT',       'FARMUSDT',      'FDUSDUSDT',      'FLOKIUSDC',
      'FLOKIUSDT',       'ICPFDUSD',        'ICPUSDT',         'IDUSDT',
         'IOUSDT',         'IQUSDT',       'JUPFDUSD',        'JUPUSDT',
       'MASKUSDT',      'MATICUSDT',       'MINAUSDT',       'NEARUSDT',
        'NFPUSDT',        'NKNUSDT',       'OMNIUSDT',     'PENDLEUSDT',
     'PEOPLEUSDT',       'PEPEUSDC',       'PEPEUSDT',       'PNUTUSDC',
     'PORTALUSDT',       'PUMPUSDC', 

In [23]:
resultados = []

for moeda in df['Pair'].unique():
    df_moeda = df[df['Pair'].isin([moeda])]
    
    df_buy = filtrar_trades(df_moeda, '2024-01-01', '2026-07-14')
    
    if df_buy.empty:
        continue  # pula moedas sem transações no período
    
    df_buy['Lucro'] = (
        df_buy['profit'] - df_buy['profit'].shift(1)
    ).where(df_buy.index % 2 == 1)
    
    df_buy['diferenca'] = (
        df_buy['Executed'] - df_buy['Executed'].shift(1)
    ).where(df_buy.index % 2 == 1)
    
    df_completos = df_buy[
        (df_buy['diferenca'] > -0.01) & (df_buy['diferenca'] < 0.0001)
    ]
    
    df_completos = df_completos.sort_values(['Pair', 'Time']).reset_index(drop=True)
    
    resultados.append(df_completos)

# Empilha tudo no final
df_final = pd.concat(resultados, ignore_index=True)
df_final.sort_values(['Time', 'Pair']).reset_index(drop=True)



,Time,Pair,Side,dia,Price,Executed,Amount,profit,Lucro,diferenca
0,2024-06-15 18:44:29,RADUSDT,SELL,2024-06-15,1.477000,67.800000,100.140600,100.140476,-95.801524,0.000000
1,2024-06-15 18:47:22,TIAUSDT,SELL,2024-06-15,7.760000,7.100000,55.096000,55.095932,-43.594068,0.000000
2,2024-06-18 15:22:20,CTSIUSDT,SELL,2024-06-18,0.157200,546.000000,85.831200,85.745369,-114.363631,0.000000
3,2024-07-03 18:49:04,PENDLEUSDT,SELL,2024-07-03,4.010000,60.700000,243.407000,243.163593,-73.690407,0.000000
4,2024-07-05 04:44:27,UMAUSDT,SELL,2024-07-05,1.852000,137.600000,254.835200,254.580365,-286.187635,0.000000
5,2024-07-05 04:45:34,FLOKIUSDT,SELL,2024-07-05,0.000132,2470000.000000,326.509300,326.182791,-0.049351,0.000000
6,2025-08-25 19:25:58,AAVEUSDC,SELL,2025-08-25,325.000000,9.434000,3066.050000,3062.983950,97.567950,-0.010000
7,2025-09-12 00:41:15,BCHUSDC,SELL,2025-09-12,600.000000,1.998000,1198.800000,1197.601200,23.201200,-0.002000
8,2025-10-10 23:19:12,AAVEUSDC,SELL,2025-10-10,148.500000,0.286000,42.471000,42.430653,-0.311444,0.000000
9,2025-10-10 23:19:12,AAVEUSDC,SELL,2025-10-10,149.370000,0.042000,6.273540,6.267580,-1.302292,-0.009000


In [24]:
df_new = df_buy[~df_buy.index.isin(df_completos.index.union(df_completos.index - 1))]
drop_columns = ['diferenca', 'Lucro']
df_new = df_new.drop(columns=drop_columns, errors='ignore')

# --- 2. Preço e quantidade de compra, ffill por par ---
df_new['preco_compra'] = df_new.groupby('Pair')['Price'].transform(
    lambda x: x.where(df_new['Side'] == 'BUY').ffill()
)
df_new['amount_compra'] = df_new.groupby('Pair')['Executed'].transform(
    lambda x: x.where(df_new['Side'] == 'BUY').ffill()
)
# --- 3. Diferença de preço simples (venda atual vs compra) ---
df_new['diferenca_preco'] = ((df_new['Price'] / df_new['preco_compra']) - 1) * 100


# --- 4. Identifica grupos: novo grupo a cada BUY ou troca de par ---
novo_grupo = (df_new['Side'] == 'BUY') | (df_new['Pair'] != df_new['Pair'].shift())
df_new['grupo'] = novo_grupo.cumsum()

df_new['executed_venda'] = df_new['Executed'].where(df_new['Side'] == 'SELL', 0)
df_new['amount_acumulado'] = df_new.groupby('grupo')['executed_venda'].cumsum()
df_new.loc[df_new['Side'] == 'BUY', 'amount_acumulado'] = 0

# --- 6. Preço médio ponderado de venda (valor vendido / qtd vendida) ---
df_new['valor_vendido'] = (df_new['Price'] * df_new['Executed']).where(df_new['Side'] == 'SELL', 0)
df_new['valor_vendido_acum'] = df_new.groupby('grupo')['valor_vendido'].cumsum()
df_new['qtd_vendida_acum'] = df_new['amount_acumulado']  # é a mesma coisa, evita recalcular
df_new['amount_buy'] = df_new.groupby('grupo')['Amount'].transform(
    lambda x: x.where(df_new['Side'] == 'BUY').ffill()
)
df_new['preco_venda_ponderado'] = df_new['valor_vendido_acum'] / df_new['qtd_vendida_acum']
drop_columns = ['grupo','executed_venda','diferenca_preco','amount_acumulado','valor_vendido','profit','preco_venda_ponderado','Executed','Price']
df_new = df_new.drop(columns=drop_columns, errors='ignore')

df_new = df_new.sort_values('Time')
df_resumo = df_new.drop_duplicates(subset=('Pair','amount_buy','preco_compra'), keep='last')
df_resumo

,Time,Pair,Side,dia,Amount,preco_compra,amount_compra,valor_vendido_acum,qtd_vendida_acum,amount_buy
2,2026-02-07 17:46:23,XRPUSDC,SELL,2026-02-07,37.788000,2.236400,447.100000,978.140000,446.600000,999.894440


In [25]:
ßßßß

NameError: name 'ßßßß' is not defined

In [ ]:


# Lucro em valor de cada venda individual
df_new['lucro_venda'] = ((df_new['Price'] - df_new['preco_compra']) * df_new['Executed']).where(df_new['Side'] == 'SELL', 0)

# Total acumulado geral (todas as vendas, todos os pares, sem resetar)
df_new['lucro_total_acumulado'] = df_new['lucro_venda'].cumsum()
# --- 8. Limpa colunas auxiliares que não precisa mais ver ---
df_new = df_new.drop(columns=['executed_venda', 'valor_vendido', 'valor_vendido_acum','grupo'], errors='ignore')
# Garante que está ordenado por tempo (caso não esteja)
df_new = df_new.sort_values('Time')

# Pega a última transação de cada Pair
df_ultima = df_new.groupby('Pair').tail(1)
df_ultima

,Time,Pair,Side,dia,Price,Executed,Amount,profit,preco_compra,amount_compra,diferenca_preco,amount_acumulado,qtd_vendida_acum,preco_venda_ponderado,lucro_venda,lucro_total_acumulado
22,2026-07-10 15:49:32,ETHEUR,SELL,2026-07-10,1580.000000,1.000000,1580.000000,1578.420000,1558.000000,2.567300,1.412067,2.536000,2.536000,1556.727392,22.000000,-3.227335


In [ ]:
df_new

,Time,Pair,Side,dia,Price,Executed,Amount,profit,preco_compra,amount_compra,diferenca_preco,amount_acumulado,qtd_vendida_acum,preco_venda_ponderado,lucro_venda,lucro_total_acumulado
12,2026-06-15 15:22:16,ETHEUR,BUY,2026-06-15,1558.000000,2.567300,3999.853400,3999.853400,1558.000000,2.567300,0.000000,0.000000,0.000000,NaN,0.000000,0.000000
13,2026-06-22 13:05:50,ETHEUR,SELL,2026-06-22,1527.000000,0.065400,99.865800,99.765934,1558.000000,2.567300,-1.989730,0.065400,0.065400,1527.000000,-2.027400,-2.027400
14,2026-06-25 11:59:11,ETHEUR,SELL,2026-06-25,1449.000000,0.041400,59.988600,59.928611,1558.000000,2.567300,-6.996149,0.106800,0.106800,1496.764045,-4.512600,-6.540000
15,2026-06-25 12:33:17,ETHEUR,SELL,2026-06-25,1445.230000,0.069100,99.865393,99.765528,1558.000000,2.567300,-7.238126,0.175900,0.175900,1476.519574,-7.792407,-14.332407
16,2026-06-26 19:10:33,ETHEUR,SELL,2026-06-26,1390.000000,0.016500,22.935000,22.912065,1558.000000,2.567300,-10.783055,0.192400,0.192400,1469.099756,-2.772000,-17.104407
17,2026-06-29 19:12:12,ETHEUR,SELL,2026-06-29,1397.000000,0.025000,34.925000,34.890075,1558.000000,2.567300,-10.333761,0.217400,0.217400,1460.808615,-4.025000,-21.129407
18,2026-06-30 01:20:42,ETHEUR,SELL,2026-06-30,1407.970000,0.047600,67.019372,66.952353,1558.000000,2.567300,-9.629653,0.265000,0.265000,1451.317604,-7.141428,-28.270835
19,2026-06-30 01:24:54,ETHEUR,SELL,2026-06-30,1406.500000,0.071000,99.861500,99.761639,1558.000000,2.567300,-9.724005,0.336000,0.336000,1441.847217,-10.756500,-39.027335
20,2026-07-03 15:41:28,ETHEUR,SELL,2026-07-03,1527.000000,0.200000,305.400000,305.094600,1558.000000,2.567300,-1.989730,0.536000,0.536000,1473.620644,-6.200000,-45.227335
21,2026-07-04 19:51:02,ETHEUR,SELL,2026-07-04,1578.000000,1.000000,1578.000000,1576.422000,1558.000000,2.567300,1.283697,1.536000,1.536000,1541.575954,20.000000,-25.227335


In [ ]:
df_new = df_buy[~df_buy.index.isin(df_completos.index.union(df_completos.index - 1))]
df_new['preco_compra'] = df_new.groupby('Pair')['Price'].transform(
    lambda x: x.where(df_new['Side'] == 'BUY').ffill()
)
drop_columns = ['ganho', 'preco_d', 'diferenca', 'Lucro']
df_new = df_new.drop(columns=drop_columns, errors='ignore')
df_new

,Time,Pair,Side,dia,Price,Executed,Amount_fe,preco_compra
0,2026-02-06 18:30:05,ETHUSDC,BUY,2026-02-06,2028.000000,1.840000,3731.515745,2028.000000
1,2026-02-06 21:47:15,ETHUSDC,SELL,2026-02-06,2055.600000,0.023800,48.923224,2028.000000
2,2026-02-06 22:33:59,ETHUSDC,SELL,2026-02-06,2060.500000,1.816200,3738.537820,2028.000000


In [ ]:
# Garante que está ordenado por tempo (caso não esteja)
df_new = df_new.sort_values('Time')

# Pega a última transação de cada Pair
df_ultima = df_new.groupby('Pair').tail(1)
df_ultima

,Time,Pair,Side,dia,Price,Executed,Amount_fe,preco_compra,amount_compra,diferenca_preco,amount_acumulado,qtd_vendida_acum,preco_venda_ponderado,retorno_ponderado,pct_vendido,lucro_venda,lucro_total_acumulado
2,2026-02-06 22:33:59,ETHUSDC,SELL,2026-02-06,2060.500000,1.816200,3738.537820,2028.000000,1.840000,1.602564,1.840000,1.840000,2060.436620,1.599439,100.000000,59.026500,59.683380


In [ ]:
df_final = pd.concat([df_completos, df_ultima], ignore_index=True)
df_final

,Time,Pair,Side,dia,Price,Executed,Amount_fe,Lucro,ganho,preco_d,...,preco_compra,amount_compra,diferenca_preco,amount_acumulado,qtd_vendida_acum,preco_venda_ponderado,retorno_ponderado,pct_vendido,lucro_venda,lucro_total_acumulado
0,2026-02-26 02:16:46,ETHEUR,SELL,2026-02-26,1750.000000,1.083000,1893.354750,138.409833,0.078868,0.080247,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-03-04 23:13:23,ETHEUR,SELL,2026-03-04,1844.000000,0.989400,1822.629146,66.445136,0.037835,0.038873,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-03-09 18:28:22,ETHEUR,SELL,2026-03-09,1752.000000,1.000300,1750.773074,0.015578,0.000009,0.004011,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-03-10 16:40:03,ETHEUR,SELL,2026-03-10,1752.000000,1.000300,1750.773074,-0.013919,-0.000008,0.006897,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-06-08 00:14:19,ETHEUR,SELL,2026-06-08,1450.000000,2.800700,4056.953985,59.023189,0.014763,0.016830,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2026-06-13 23:42:19,ETHEUR,SELL,2026-06-13,1465.000000,2.740600,4010.964021,11.089564,0.002772,0.004801,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,2026-03-13 16:34:27,ETHUSDC,BUY,2026-03-13,2195.500000,2.787800,6120.612112,0.098752,0.000016,0.001597,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,2026-03-25 09:47:25,ETHUSDC,BUY,2026-03-25,2163.000000,2.792500,6040.174707,0.164763,0.000027,0.001389,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2026-04-20 01:33:27,ETHUSDC,SELL,2026-04-20,2266.000000,2.417400,5472.350572,-428.807509,-0.072665,-0.070931,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,2026-04-21 12:50:02,ETHUSDC,SELL,2026-04-21,2337.000000,2.354300,5496.497101,24.242058,0.004430,0.006460,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
